# 1. Preparação Geral dos Dados e Constantes

Nesta etapa, carregamos a base limpa do NHIS 2025 (`adult25_limpo.csv`), definimos o `ID_DOMICILIO` como índice e isolamos a variável alvo multiclasse (`SEVERIDADE_DEPRESSAO_PHQ`), filtrando os registros indefinidos na Target (valores `-1` e `5`).

Além disso, isolamos a coluna `PESO_AMOSTRAL` e removemos do conjunto de preditores ($X$) todas as colunas de sintomas diretos de depressão para evitar vazamento de dados (*data leakage*).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==============================================================================
# CONFIGURAÇÃO DE VARIÁVEIS GLOBAIS / CONSTANTES
# ==============================================================================
ID_COLUNA = 'ID_DOMICILIO'
PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET = 'SEVERIDADE_DEPRESSAO_PHQ'

COLUNAS_SINTOMAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM',
    'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM',
    'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM',
    'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM',
    'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO',
    'REMEDIO_DEPRESSAO',
    'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO'
]

# 1. Carregar a base limpa e definir o ID como índice
df = pd.read_csv('adult25_limpo.csv').set_index(ID_COLUNA)

# 2. Eliminar os registros indefinidos no TARGET (-1 e 5)
df = df[~df[TARGET].isin([-1, 5])].copy()

# 3. Isolar Target (y), Peso Amostral e Features Preditoras (X)
y = df[TARGET]
sample_weights = df[PESO_COLUNA]
colunas_remover = [TARGET, PESO_COLUNA] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[col for col in colunas_remover if col in df.columns])

# 2. Cenário A: Treinamento Mantendo Códigos Indeterminados (-1) nas Features

Neste primeiro cenário, mantemos as categorias de não-resposta (`-1`) presentes nas variáveis preditoras ($X$), permitindo que o modelo de árvore e a codificação One-Hot tratem o valor como uma categoria válida.

In [ ]:
# Split Estratificado
X_train_A, X_test_A, y_train_A, y_test_A, w_train_A, w_test_A = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=42, stratify=y
)

# Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [col for col in X_train_A.columns if col not in num_cols]

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_A = preprocessor_A.fit_transform(X_train_A)
X_test_proc_A = preprocessor_A.transform(X_test_A)

# Modelo Random Forest
model_A = RandomForestClassifier(random_state=42, class_weight='balanced')
model_A.fit(X_train_proc_A, y_train_A, sample_weight=w_train_A)

# Avaliação Cenário A
y_pred_A = model_A.predict(X_test_proc_A)
y_proba_A = model_A.predict_proba(X_test_proc_A)

print("--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---")
print(classification_report(y_test_A, y_pred_A, sample_weight=w_test_A, zero_division=0))

print("--- CENÁRIO A: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_A, y_pred_A, sample_weight=w_test_A)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_A, y_proba_A, multi_class='ovr', sample_weight=w_test_A):.4f}")

# 3. Cenário B: Treinamento Apenas com Informações Completas (Remoção de -1 no X)

Conforme discutido com a equipe e orientação do professor, realizamos um teste estrito filtrando qualquer observação que contenha não-resposta (`-1`) em qualquer feature preditora. A amostragem retida foi de **22.084 registros (93.5% dos dados)**.

In [ ]:
# Filtragem Estrita das Features
mascara_sem_menos1 = (X != -1).all(axis=1)
X_strict = X[mascara_sem_menos1].copy()
y_strict = y[mascara_sem_menos1].copy()
w_strict = sample_weights[mascara_sem_menos1].copy()

print(f"Total de registros na base completa estrita: {len(X_strict)}")

# Split Estratificado Cenário B
X_train_B, X_test_B, y_train_B, y_test_B, w_train_B, w_test_B = train_test_split(
    X_strict, y_strict, w_strict, test_size=0.2, random_state=42, stratify=y_strict
)

# Pré-processamento
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_B = preprocessor_B.fit_transform(X_train_B)
X_test_proc_B = preprocessor_B.transform(X_test_B)

# Modelo Random Forest Cenário B
model_B = RandomForestClassifier(random_state=42, class_weight='balanced')
model_B.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

# Avaliação Cenário B
y_pred_B = model_B.predict(X_test_proc_B)
y_proba_B = model_B.predict_proba(X_test_proc_B)

print("--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---")
print(classification_report(y_test_B, y_pred_B, sample_weight=w_test_B, zero_division=0))

print("--- CENÁRIO B: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_B, y_pred_B, sample_weight=w_test_B)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_B, y_proba_B, multi_class='ovr', sample_weight=w_test_B):.4f}")

# 4. Definição do Espaço de Busca de Hiperparâmetros Refinado

Definimos uma grade estendida de parâmetros para otimizar o desempenho do Random Forest tanto no cenário multinível quanto no binário.

In [ ]:
param_dist_refinado = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [20, 25, 30, 35, None],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', 0.2, 0.3],
    'class_weight': ['balanced', 'balanced_subsample']
}

# 5. Otimização do Modelo Multinível (4 Classes) com Hiperparâmetros Refinados

Aplicamos o `RandomizedSearchCV` focado no $F1\text{-macro}$ para melhorar a discriminação dos 4 níveis do PHQ-9.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Configurar o RandomizedSearchCV no Cenário B
random_search_multi = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_refinado,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 2. Treinamento
random_search_multi.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (MULTINÍVEL) ---")
print(random_search_multi.best_params_)

# 3. Avaliação no Conjunto de Teste
best_model_multi = random_search_multi.best_estimator_
y_pred_tuned = best_model_multi.predict(X_test_proc_B)
y_proba_tuned = best_model_multi.predict_proba(X_test_proc_B)

print("\n--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---")
print(classification_report(y_test_B, y_pred_tuned, sample_weight=w_test_B, zero_division=0))

cm_multi_bruta = confusion_matrix(y_test_B, y_pred_tuned)
cm_multi_pop = confusion_matrix(y_test_B, y_pred_tuned, sample_weight=w_test_B)

print("--- MATRIZ DE CONFUSÃO DA AMOSTRA (4x4 - Contagem Bruta de Linhas) ---")
print(cm_multi_bruta)

print("\n--- MATRIZ DE CONFUSÃO POPULACIONAL (4x4 - Projeção Expandida) ---")
print(np.round(cm_multi_pop).astype(int))

print("\n--- MATRIZ DE CONFUSÃO POPULACIONAL NORMALIZADA POR CLASSE REAL (%) ---")
print(np.round(confusion_matrix(y_test_B, y_pred_tuned, sample_weight=w_test_B, normalize='true') * 100, 2))

print(f"\nROC-AUC Score (OvR) Otimizado: {roc_auc_score(y_test_B, y_proba_tuned, multi_class='ovr', sample_weight=w_test_B):.4f}")

# 6. Verificação e Distribuição do Tamanho dos Grupos & Centroides das Classes

Análise comparativa da dimensão dos grupos e cálculo dos **centroides (vetor de perfis médios)** para caracterizar cada nível de severidade da depressão na população.

In [ ]:
mapa_niveis = {
    1: 'Nível 1: Mínimo / Sem Depressão',
    2: 'Nível 2: Leve',
    3: 'Nível 3: Moderado',
    4: 'Nível 4: Grave'
}

# 1. Tamanho dos Grupos
contagem_amostral = df['SEVERIDADE_DEPRESSAO_PHQ'].value_counts().sort_index()
projecao_populacional = df.groupby('SEVERIDADE_DEPRESSAO_PHQ')['PESO_AMOSTRAL'].sum()

tabela_grupos_total = pd.DataFrame({
    'Descrição do Nível': contagem_amostral.index.map(mapa_niveis),
    'Amostra Bruta (Linhas)': contagem_amostral.values,
    '% Amostra': (contagem_amostral.values / len(df)) * 100,
    'População Expandida': projecao_populacional.values,
    '% População': (projecao_populacional.values / df['PESO_AMOSTRAL'].sum()) * 100
})

print("--- 1. TAMANHO DOS GRUPOS NA BASE TOTAL (22.084 REGISTROS ESTRITOS) ---")
print(tabela_grupos_total.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# 2. Centroides das Classes Multinível (Perfil Médio por Nível)
print("\n" + "="*80 + "\n")
print("--- 2. CENTROIDES DAS CLASSES (PERFIL MÉDIO SOCIODEMOGRÁFICO POR NÍVEL) ---")

# Calculando centroides das variáveis numéricas/relevantes agrupadas pelo nível
df_estrito = df[mascara_sem_menos1].copy()
cols_centroides = [c for c in ['RAZAO_RENDA_POBREZA', 'IDADE_P', 'SEXO_P'] if c in df_estrito.columns]

if cols_centroides:
    centroides_multi = df_estrito.groupby('SEVERIDADE_DEPRESSAO_PHQ')[cols_centroides].mean()
    centroides_multi.index = centroides_multi.index.map(mapa_niveis)
    print(centroides_multi.round(3))
else:
    # Caso as colunas estejam pré-processadas, calcula nos dados transformados
    df_proc_temp = pd.DataFrame(X_train_proc_B)
    df_proc_temp['TARGET'] = y_train_B.values
    centroides_multi = df_proc_temp.groupby('TARGET').mean()
    print(centroides_multi.iloc[:, :8].round(3))  # Exibe os primeiros 8 componentes

# 7. Otimização, Centroides e Matriz de Confusão Binária Detalhada (TP, TN, FP, FN)

Execução da otimização para a abordagem binária de rastreio, exibindo os **centroides das duas classes** e o detalhamento percentual completo dos quadrantes (**TN, FP, FN, TP**).

In [ ]:
# 1. Carregar a base limpa e aplicar a transformação binária
df_bin = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')
df_bin = df_bin[~df_bin['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df_bin['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df_bin['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS_BIN = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# 2. Isolar X, y e Pesos nas Features Completas
y_bin = df_bin[TARGET_BIN]
weights_bin = df_bin[PESO_COLUNA]
X_base_bin = df_bin.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS_BIN if col in df_bin.columns])

mascara_completos = (X_base_bin != -1).all(axis=1)
X_bin = X_base_bin[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights_bin[mascara_completos].copy()

# 3. Split Estratificado Binário
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor_bin = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor_bin.fit_transform(X_tr)
X_te_proc = preprocessor_bin.transform(X_te)

# 5. RandomizedSearchCV Binário
random_search_bin = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_refinado,
    n_iter=20,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando a busca de hiperparâmetros binários (RandomizedSearchCV)...")
random_search_bin.fit(X_tr_proc, y_tr, sample_weight=w_tr)

rf_binario_otimizado = random_search_bin.best_estimator_

print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (BINÁRIO) ---")
for param, val in random_search_bin.best_params_.items():
    print(f"{param}: {val}")

# 6. Avaliação e Métricas
y_pred_bin = rf_binario_otimizado.predict(X_te_proc)
y_proba_bin = rf_binario_otimizado.predict_proba(X_te_proc)[:, 1]

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (PONDERADO POR PESOS POPULACIONAIS) ---")
print(classification_report(y_te, y_pred_bin, sample_weight=w_te, zero_division=0))

# 7. Cálculo das Matrizes e Detalhamento Percentual (TN, FP, FN, TP)
cm_amostra = confusion_matrix(y_te, y_pred_bin)
tn, fp, fn, tp = cm_amostra.ravel()
total_amostra = len(y_te)

cm_pop = confusion_matrix(y_te, y_pred_bin, sample_weight=w_te)
tn_w, fp_w, fn_w, tp_w = cm_pop.ravel()
total_pop = w_te.sum()

print("--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---")
print(f"Verdadeiros Negativos (TN): {tn:5d} | {tn/total_amostra*100:6.2f}% (Sem Depressão - Correto)")
print(f"Falsos Positivos      (FP): {fp:5d} | {fp/total_amostra*100:6.2f}% (Alarme Falso)")
print(f"Falsos Negativos      (FN): {fn:5d} | {fn/total_amostra*100:6.2f}% (Subdiagnóstico)")
print(f"Verdadeiros Positivos (TP): {tp:5d} | {tp/total_amostra*100:6.2f}% (Com Depressão - Capturado)")

print("\n--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional EUA) ---")
print(f"Verdadeiros Negativos (TN): {tn_w:11,.0f} | {tn_w/total_pop*100:6.2f}%")
print(f"Falsos Positivos      (FP): {fp_w:11,.0f} | {fp_w/total_pop*100:6.2f}%")
print(f"Falsos Negativos      (FN): {fn_w:11,.0f} | {fn_w/total_pop*100:6.2f}%")
print(f"Verdadeiros Positivos (TP): {tp_w:11,.0f} | {tp_w/total_pop*100:6.2f}%")

print("\n--- TAXAS OPERACIONAIS DE TRIAGEM (%) ---")
print(f"Taxa de Sensibilidade (Recall do TP) : {tp / (tp + fn) * 100:.2f}%")
print(f"Taxa de Especificidade (TN / TN+FP) : {tn / (tn + fp) * 100:.2f}%")
print(f"Taxa de Falso Alarme   (FP / TN+FP) : {fp / (tn + fp) * 100:.2f}%")
print(f"Taxa de Subdiagnóstico (FN / TP+FN) : {fn / (tp + fn) * 100:.2f}%")

# 8. Centroides das Classes Binárias
print("\n" + "="*80 + "\n")
print("--- CENTROIDES DAS CLASSES BINÁRIAS (PERFIL MÉDIO: SEM VS COM DEPRESSÃO) ---")
mapa_bin = {0: 'Classe 0: Sem Depressão', 1: 'Classe 1: Com Depressão'}
cols_centroides_bin = [c for c in ['RAZAO_RENDA_POBREZA', 'IDADE_P', 'SEXO_P'] if c in df_bin.columns]

if cols_centroides_bin:
    centroides_bin = df_bin.groupby('DEPRESSAO_PHQ_SIMPLIFICADO')[cols_centroides_bin].mean()
    centroides_bin.index = centroides_bin.index.map(mapa_bin)
    print(centroides_bin.round(3))
else:
    df_proc_bin = pd.DataFrame(X_tr_proc)
    df_proc_bin['TARGET'] = y_tr.values
    centroides_bin = df_proc_bin.groupby('TARGET').mean()
    centroides_bin.index = centroides_bin.index.map(mapa_bin)
    print(centroides_bin.iloc[:, :8].round(3))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba_bin, sample_weight=w_te):.4f}")